# IAN 603 · Module 5
# Reasoning with SQL Logic

Build reliable SQL rules with Boolean logic, NULL-aware reasoning, and CASE expressions.

Course module:  
https://ali2449.github.io/IAN-603-Preparing-Data-for-Analytics-/s_module5.html


## Learning goals

Work through the examples in order. Focus on the meaning of each row, the reason for each SQL clause, and how you would validate the result.


## 0. Install required Python packages

Run this only if the packages are not already installed.


In [ ]:
%pip install -q pandas sqlalchemy psycopg2-binary ipywidgets


## 1. Load the course database helper

This notebook uses a small course helper to open the PostgreSQL connection window and display query results.

**You only need this notebook file.** When you run the next cell, the helper is loaded automatically from the course GitHub repository if it is not already available.

If you would like your own copy of the helper, you can download it here:

[Download `ian603_database.py`](https://github.com/ali2449/IAN-603-Preparing-Data-for-Analytics-/raw/refs/heads/MSIA/ian603_database.py)

After the helper loads, continue to the connection section below.


In [ ]:
from pathlib import Path
import urllib.request

MODULE_FILE = Path("ian603_database.py")
MODULE_URL = (
    "https://raw.githubusercontent.com/"
    "ali2449/IAN-603-Preparing-Data-for-Analytics-/MSIA/"
    "ian603_database.py"
)

if not MODULE_FILE.exists():
    try:
        urllib.request.urlretrieve(MODULE_URL, MODULE_FILE)
        print("Course database helper downloaded.")
    except Exception as e:
        print("The database helper could not be downloaded automatically.")
        print("Use the download link in the Markdown cell above, then run this cell again.")
        raise

from ian603_database import IAN603Database

db = IAN603Database()

display_query = db.display_query
execute_query = db.execute_query
execute_script = db.execute_script

print("Database helper ready.")


## 2. Connect to PostgreSQL

Run the next cell to open the connection window. It works much like the connection settings you use in pgAdmin.

The usual local PostgreSQL defaults are already filled in:

| Setting | Default |
|---|---|
| Host | `localhost` |
| Database | `postgres` |
| Username | `postgres` |
| Port | `5432` |

Enter your PostgreSQL password, then click **Connect**.

If you are using your own computer, you may also choose to save the connection settings for the next time you open the notebook.


In [ ]:
db.show_connection_panel()


## 3. Check the connection

After you click **Connect**, run the next cell to make sure the notebook is ready to use PostgreSQL.


In [ ]:
if db.connected:
    db.test_connection()
else:
    print("No active PostgreSQL connection.")
    print("Return to the connection panel above and click Connect.")


## How to think about Module 5

SQL logic has three possible truth states:

| Expression result | WHERE behavior |
|---|---|
| TRUE | row is kept |
| FALSE | row is removed |
| UNKNOWN | row is removed |

`NULL` often creates **UNKNOWN**, which is why:

```sql
column = NULL
```

does not work as intended.

Use:

```sql
column IS NULL
```

### Rule-building workflow

1. Define the business rule in plain language.
2. Identify NULL and blank cases.
3. Group `AND` / `OR` conditions with parentheses.
4. Order `CASE` branches from specific to general.
5. Test edge cases.
6. Validate the result.


# Teaching database

Run the next cell to create or reset the Module 5 teaching data. You can return to this cell whenever you want a clean starting point.


In [ ]:
setup_sql = "-- ============================================================\n-- IAN 603 - MODULE 5\n-- Reasoning with SQL Logic\n-- ============================================================\n-- Companion file for s_module5.html\n-- PostgreSQL / pgAdmin Query Tool\n--\n-- Topics:\n-- AND, OR, NOT, operator precedence, three-valued logic,\n-- NULL handling, IS TRUE / FALSE / UNKNOWN, IS DISTINCT FROM,\n-- COALESCE, NULLIF, CASE, and applied decision rules.\n--\n-- Important: WHERE keeps only TRUE. FALSE and UNKNOWN rows are\n-- both filtered out unless the rule explicitly handles them.\n-- ============================================================\n\n-- ============================================================\n-- SECTION 1 - CREATE / RESET TEACHING DATABASE\n-- ============================================================\n\nDROP VIEW IF EXISTS patient_case CASCADE;\nDROP TABLE IF EXISTS admissions CASCADE;\nDROP TABLE IF EXISTS patient_billing CASCADE;\nDROP TABLE IF EXISTS diagnostics CASCADE;\nDROP TABLE IF EXISTS patient_info CASCADE;\n\nCREATE TABLE patient_info (\n    patient_id integer PRIMARY KEY,\n    patient_name varchar(100) NOT NULL,\n    age integer NOT NULL CHECK (age BETWEEN 0 AND 120),\n    email_address varchar(150),\n    mobile_phone varchar(30),\n    home_phone varchar(30),\n    active boolean NOT NULL DEFAULT true,\n    consent_to_contact boolean,\n    preferred_language varchar(40)\n);\n\nINSERT INTO patient_info\n    (patient_id, patient_name, age, email_address, mobile_phone, home_phone, active, consent_to_contact, preferred_language)\nVALUES\n    (101, 'John Doe', 65, 'john.doe@email.com', '336-555-0101', NULL, true, true, 'English'),\n    (102, 'Jane Smith', 32, 'jane.smith@email.com', '336-555-0102', NULL, true, false, 'English'),\n    (103, 'Robert Lee', 48, NULL, NULL, '336-555-0103', true, true, 'English'),\n    (104, 'Lisa Ray', 71, '', NULL, '336-555-0104', true, false, 'English'),\n    (105, 'Peter Chen', 25, 'peter.chen@email.com', '336-555-0105', NULL, false, true, 'Mandarin'),\n    (106, 'Susan Gray', 58, 'susan.gray@email.com', NULL, NULL, true, NULL, NULL),\n    (107, 'Maria Torres', 44, '   ', '336-555-0107', NULL, true, true, 'Spanish'),\n    (108, 'Andre Green', 16, NULL, NULL, NULL, true, false, 'English');\n\nCREATE TABLE diagnostics (\n    patient_id integer PRIMARY KEY REFERENCES patient_info(patient_id),\n    diagnosis varchar(100),\n    severity varchar(20) CHECK (severity IN ('Low', 'Moderate', 'High')),\n    confirmed boolean\n);\nINSERT INTO diagnostics (patient_id, diagnosis, severity, confirmed) VALUES\n    (101, 'Pneumonia', 'High', true), (102, 'Fever', 'Moderate', true),\n    (103, 'Kidney Disease', 'High', true), (104, 'Diabetes', 'High', true),\n    (105, 'Sprained Ankle', 'Low', true), (106, 'Kidney Disease', 'High', NULL),\n    (107, 'Asthma', 'Moderate', true), (108, 'Flu', 'Moderate', false);\n\nCREATE TABLE patient_billing (\n    patient_id integer PRIMARY KEY REFERENCES patient_info(patient_id),\n    bill_amount numeric(10,2) CHECK (bill_amount IS NULL OR bill_amount >= 0),\n    amount_paid numeric(10,2) CHECK (amount_paid IS NULL OR amount_paid >= 0),\n    payment_status varchar(30)\n);\nINSERT INTO patient_billing (patient_id, bill_amount, amount_paid, payment_status) VALUES\n    (101, 12000.00, 4000.00, 'Open'), (102, 7500.00, 7500.00, 'Paid'),\n    (103, NULL, 0.00, 'Pending Review'), (104, 9800.00, NULL, 'Open'),\n    (105, 850.00, 850.00, 'Paid'), (106, 5200.00, 1000.00, NULL),\n    (107, 0.00, 0.00, 'No Charge'), (108, 2300.00, 0.00, 'Open');\n\nCREATE TABLE admissions (\n    admission_id integer PRIMARY KEY,\n    patient_id integer NOT NULL REFERENCES patient_info(patient_id),\n    admission_date date NOT NULL,\n    discharge_date date,\n    diagnosis varchar(100) NOT NULL\n);\nINSERT INTO admissions (admission_id, patient_id, admission_date, discharge_date, diagnosis) VALUES\n    (201, 101, '2026-07-15', '2026-07-21', 'Pneumonia'),\n    (202, 102, '2026-07-25', '2026-07-27', 'Fever'),\n    (203, 103, '2026-08-01', NULL, 'Kidney Disease'),\n    (204, 104, '2026-06-10', '2026-06-22', 'Diabetes'),\n    (205, 105, '2026-08-05', '2026-08-06', 'Sprained Ankle'),\n    (206, 106, '2026-07-30', '2026-08-04', 'Kidney Disease'),\n    (207, 107, '2026-08-08', NULL, 'Asthma'),\n    (208, 108, '2026-08-09', '2026-08-11', 'Flu');\n\nCREATE VIEW patient_case AS\nSELECT p.patient_id, p.patient_name, p.age, p.email_address, p.mobile_phone, p.home_phone,\n       p.active, p.consent_to_contact, p.preferred_language,\n       d.diagnosis, d.severity, d.confirmed,\n       b.bill_amount, b.amount_paid, b.payment_status\nFROM patient_info AS p\nLEFT JOIN diagnostics AS d USING (patient_id)\nLEFT JOIN patient_billing AS b USING (patient_id);"

execute_script(setup_sql)


# INSPECT EDGE CASES


## INSPECT EDGE CASES · Step 1

Run this step and connect the result back to the module concept.


In [ ]:
query = "SELECT * FROM patient_info ORDER BY patient_id;"

display_query(query, title="INSPECT EDGE CASES · Step 1")


## INSPECT EDGE CASES · Step 2

Run this step and connect the result back to the module concept.


In [ ]:
query = "SELECT * FROM diagnostics ORDER BY patient_id;"

display_query(query, title="INSPECT EDGE CASES · Step 2")


## INSPECT EDGE CASES · Step 3

Run this step and connect the result back to the module concept.


In [ ]:
query = "SELECT * FROM patient_billing ORDER BY patient_id;"

display_query(query, title="INSPECT EDGE CASES · Step 3")


## INSPECT EDGE CASES · Step 4

Run this step and connect the result back to the module concept.


In [ ]:
query = "SELECT * FROM admissions ORDER BY admission_id;"

display_query(query, title="INSPECT EDGE CASES · Step 4")


# RUNNABLE MODULE EXAMPLES


## Example 01

Run the SQL, inspect the output carefully, and explain what one output row represents.


In [ ]:
query = "SELECT patient_id, patient_name, age, diagnosis, severity\nFROM patient_case\nWHERE active IS TRUE\n  AND age >= 40\n  AND severity = 'High'\nORDER BY patient_id;"

display_query(query, title="Example 01")


## Example 02

Run the SQL, inspect the output carefully, and explain what one output row represents.


In [ ]:
query = "SELECT patient_id, patient_name, diagnosis\nFROM patient_case\nWHERE diagnosis = 'Pneumonia'\n   OR diagnosis = 'Flu'\n   OR diagnosis = 'Asthma'\nORDER BY patient_id;"

display_query(query, title="Example 02")


## Example 03

Run the SQL, inspect the output carefully, and explain what one output row represents.


In [ ]:
query = "SELECT patient_id, patient_name, diagnosis, active\nFROM patient_case\nWHERE active IS TRUE\n  AND NOT (diagnosis IN ('Sprained Ankle', 'Flu'))\nORDER BY patient_id;"

display_query(query, title="Example 03")


## Example 04

Run the SQL, inspect the output carefully, and explain what one output row represents.


In [ ]:
query = "-- Version A: AND binds before OR\nSELECT patient_id, patient_name, age, diagnosis, consent_to_contact\nFROM patient_case\nWHERE age >= 65\n   OR diagnosis = 'Kidney Disease' AND consent_to_contact IS TRUE\nORDER BY patient_id;\n\n-- Version B: consent applies to the whole rule\nSELECT patient_id, patient_name, age, diagnosis, consent_to_contact\nFROM patient_case\nWHERE (age >= 65 OR diagnosis = 'Kidney Disease')\n  AND consent_to_contact IS TRUE\nORDER BY patient_id;"

execute_script(query)


## Example 05

Run the SQL, inspect the output carefully, and explain what one output row represents.


In [ ]:
query = "WITH bools(label, value, sort_order) AS (\n    VALUES\n        ('TRUE', TRUE, 1),\n        ('FALSE', FALSE, 2),\n        ('UNKNOWN', NULL::boolean, 3)\n)\nSELECT a.label AS a,\n       b.label AS b,\n       COALESCE((a.value AND b.value)::text, 'unknown') AS a_and_b,\n       COALESCE((a.value OR b.value)::text, 'unknown') AS a_or_b\nFROM bools AS a\nCROSS JOIN bools AS b\nORDER BY a.sort_order, b.sort_order;"

display_query(query, title="Example 05")


## Example 06

Run the SQL, inspect the output carefully, and explain what one output row represents.


In [ ]:
query = "SELECT label, flag\nFROM (VALUES\n    ('TRUE row', TRUE),\n    ('FALSE row', FALSE),\n    ('UNKNOWN row', NULL::boolean)\n) AS states(label, flag)\nWHERE flag;"

display_query(query, title="Example 06")


## Example 07

Run the SQL, inspect the output carefully, and explain what one output row represents.


In [ ]:
query = "SELECT patient_id,\n       email_address,\n       email_address = NULL AS equals_null,\n       email_address <> NULL AS not_equals_null\nFROM patient_info\nORDER BY patient_id;"

display_query(query, title="Example 07")


## Example 08

Run the SQL, inspect the output carefully, and explain what one output row represents.


In [ ]:
query = "SELECT patient_id, patient_name, email_address\nFROM patient_info\nWHERE NULLIF(BTRIM(email_address), '') IS NULL\nORDER BY patient_id;"

display_query(query, title="Example 08")


## Example 09

Run the SQL, inspect the output carefully, and explain what one output row represents.


In [ ]:
query = "SELECT patient_id, patient_name, consent_to_contact,\n       consent_to_contact IS TRUE AS explicitly_yes,\n       consent_to_contact IS FALSE AS explicitly_no,\n       consent_to_contact IS UNKNOWN AS not_recorded\nFROM patient_info\nORDER BY patient_id;"

display_query(query, title="Example 09")


## Example 10

Run the SQL, inspect the output carefully, and explain what one output row represents.


In [ ]:
query = "SELECT patient_id, bill_amount, amount_paid,\n       bill_amount IS DISTINCT FROM amount_paid AS amounts_differ,\n       bill_amount IS NOT DISTINCT FROM amount_paid AS amounts_match_null_safely\nFROM patient_billing\nORDER BY patient_id;"

display_query(query, title="Example 10")


## Example 11

Run the SQL, inspect the output carefully, and explain what one output row represents.


In [ ]:
query = "SELECT patient_id,\n       email_address AS original_email,\n       NULLIF(BTRIM(email_address), '') AS normalized_email\nFROM patient_info\nORDER BY patient_id;"

display_query(query, title="Example 11")


## Example 12

Run the SQL, inspect the output carefully, and explain what one output row represents.


In [ ]:
query = "SELECT patient_id,\n       patient_name,\n       COALESCE(\n           NULLIF(BTRIM(email_address), ''),\n           NULLIF(BTRIM(mobile_phone), ''),\n           NULLIF(BTRIM(home_phone), ''),\n           'No contact provided'\n       ) AS preferred_contact\nFROM patient_info\nORDER BY patient_id;"

display_query(query, title="Example 12")


## Example 13

Run the SQL, inspect the output carefully, and explain what one output row represents.


In [ ]:
query = "SELECT patient_id, bill_amount, amount_paid,\n       ROUND(\n           100.0 * COALESCE(amount_paid, 0) / NULLIF(bill_amount, 0),\n           1\n       ) AS percent_paid\nFROM patient_billing\nORDER BY patient_id;"

display_query(query, title="Example 13")


## Example 14

Run the SQL, inspect the output carefully, and explain what one output row represents.


In [ ]:
query = "SELECT patient_id, bill_amount,\n       CASE\n           WHEN bill_amount IS NULL THEN 'Unknown'\n           WHEN bill_amount >= 10000 THEN 'High-Value'\n           WHEN bill_amount >= 5000 THEN 'Standard'\n           ELSE 'Low-Value'\n       END AS billing_tier\nFROM patient_billing\nORDER BY patient_id;"

display_query(query, title="Example 14")


## Example 15

Run the SQL, inspect the output carefully, and explain what one output row represents.


In [ ]:
query = "SELECT patient_id, payment_status,\n       CASE payment_status\n           WHEN 'Paid' THEN 'Closed'\n           WHEN 'No Charge' THEN 'Closed'\n           WHEN 'Open' THEN 'Follow up'\n           WHEN 'Pending Review' THEN 'Investigate'\n           ELSE 'Status missing'\n       END AS action\nFROM patient_billing\nORDER BY patient_id;"

display_query(query, title="Example 15")


## Example 16

Run the SQL, inspect the output carefully, and explain what one output row represents.


In [ ]:
query = "-- Wrong order: senior branch is unreachable\nSELECT patient_id, patient_name, age,\n       CASE\n           WHEN age >= 18 THEN 'Adult'\n           WHEN age >= 65 THEN 'Senior'\n           ELSE 'Minor'\n       END AS age_group\nFROM patient_info\nORDER BY patient_id;\n\n-- Correct order\nSELECT patient_id, patient_name, age,\n       CASE\n           WHEN age >= 65 THEN 'Senior'\n           WHEN age >= 18 THEN 'Adult'\n           ELSE 'Minor'\n       END AS age_group\nFROM patient_info\nORDER BY patient_id;"

execute_script(query)


## Example 17

Run the SQL, inspect the output carefully, and explain what one output row represents.


In [ ]:
query = "SELECT patient_id, diagnosis,\n       CASE\n           WHEN diagnosis IN ('Pneumonia', 'Flu', 'Asthma')\n               THEN 'Respiratory'\n       END AS diagnosis_group\nFROM diagnostics\nORDER BY patient_id;"

display_query(query, title="Example 17")


## Example 18

Run the SQL, inspect the output carefully, and explain what one output row represents.


In [ ]:
query = "SELECT patient_id, payment_status\nFROM patient_billing\nWHERE payment_status NOT IN ('Paid', NULL)\nORDER BY patient_id;"

display_query(query, title="Example 18")


## Example 19

Run the SQL, inspect the output carefully, and explain what one output row represents.


In [ ]:
query = "-- Ordinary inequality excludes NULL status\nSELECT patient_id, payment_status\nFROM patient_billing\nWHERE payment_status <> 'Paid'\nORDER BY patient_id;\n\n-- NULL-safe inequality intentionally includes missing status\nSELECT patient_id, payment_status\nFROM patient_billing\nWHERE payment_status IS DISTINCT FROM 'Paid'\nORDER BY patient_id;"

execute_script(query)


## Example 20

Run the SQL, inspect the output carefully, and explain what one output row represents.


In [ ]:
query = "SELECT patient_id, patient_name, diagnosis, consent_to_contact, email_address\nFROM patient_case\nWHERE active IS TRUE\n  AND consent_to_contact IS TRUE\n  AND diagnosis IN ('Diabetes', 'Kidney Disease')\n  AND NULLIF(BTRIM(email_address), '') IS NULL\nORDER BY patient_id;"

display_query(query, title="Example 20")


## Example 21

Run the SQL, inspect the output carefully, and explain what one output row represents.


In [ ]:
query = "SELECT patient_id, patient_name,\n       COALESCE(NULLIF(BTRIM(email_address), ''),\n                NULLIF(BTRIM(mobile_phone), ''),\n                NULLIF(BTRIM(home_phone), ''),\n                'No contact provided') AS preferred_contact,\n       bill_amount,\n       CASE\n           WHEN bill_amount IS NULL THEN 'Needs review'\n           WHEN bill_amount = 0 THEN 'No Charge'\n           WHEN bill_amount >= 10000 THEN 'High-Value'\n           WHEN bill_amount >= 5000 THEN 'Standard'\n           ELSE 'Low-Value'\n       END AS billing_tier\nFROM patient_case\nORDER BY patient_id;"

display_query(query, title="Example 21")


## Example 22

Run the SQL, inspect the output carefully, and explain what one output row represents.


In [ ]:
query = "SELECT admission_id, patient_id, admission_date, discharge_date,\n       discharge_date - admission_date AS length_of_stay_days,\n       CASE\n           WHEN discharge_date - admission_date < 5 THEN 'Short'\n           WHEN discharge_date - admission_date <= 10 THEN 'Medium'\n           ELSE 'Long'\n       END AS stay_category\nFROM admissions\nWHERE discharge_date IS NOT NULL\nORDER BY admission_id;"

display_query(query, title="Example 22")


## Example 23

Run the SQL, inspect the output carefully, and explain what one output row represents.


In [ ]:
query = "SELECT a.admission_id, p.patient_name, p.age, a.admission_date, a.diagnosis,\n       CASE\n           WHEN a.diagnosis IN ('Pneumonia', 'Flu') AND p.age >= 60\n               THEN 'High-Risk Respiratory'\n           WHEN a.diagnosis IN ('Pneumonia', 'Flu')\n               THEN 'Respiratory'\n           ELSE 'Other'\n       END AS diagnosis_category\nFROM admissions AS a\nJOIN patient_info AS p USING (patient_id)\nWHERE a.admission_date BETWEEN DATE '2026-08-11' - 30 AND DATE '2026-08-11'\n  AND a.diagnosis IN ('Pneumonia', 'Flu')\nORDER BY a.admission_date;"

display_query(query, title="Example 23")


## Example 24

Run the SQL, inspect the output carefully, and explain what one output row represents.


In [ ]:
query = "SELECT patient_id, patient_name, preferred_language,\n       COALESCE(NULLIF(BTRIM(preferred_language), ''), 'Language not recorded') AS language_display\nFROM patient_info\nORDER BY patient_id;"

display_query(query, title="Example 24")


## Example 25

Run the SQL, inspect the output carefully, and explain what one output row represents.


In [ ]:
query = "SELECT patient_id, bill_amount,\n       CASE\n           WHEN bill_amount IS NULL THEN 'Unknown'\n           WHEN bill_amount = 0 THEN 'No Charge'\n           WHEN bill_amount >= 10000 THEN 'High-Value'\n           WHEN bill_amount >= 5000 THEN 'Standard'\n           ELSE 'Low-Value'\n       END AS billing_tier\nFROM patient_billing\nORDER BY patient_id;"

display_query(query, title="Example 25")


# NULL / BLANK NORMALIZATION PATTERNS


## NULL / BLANK NORMALIZATION PATTERNS · Step 1

Run this step and connect the result back to the module concept.


In [ ]:
query = "SELECT\n    patient_id,\n    email_address AS raw_email,\n    NULLIF(BTRIM(email_address), '') AS normalized_email\nFROM patient_info\nORDER BY patient_id;"

display_query(query, title="NULL / BLANK NORMALIZATION PATTERNS · Step 1")


## NULL / BLANK NORMALIZATION PATTERNS · Step 2

Run this step and connect the result back to the module concept.


In [ ]:
query = "SELECT\n    patient_id,\n    patient_name,\n    COALESCE(\n        NULLIF(BTRIM(email_address), ''),\n        NULLIF(BTRIM(mobile_phone), ''),\n        NULLIF(BTRIM(home_phone), ''),\n        'No contact provided'\n    ) AS preferred_contact\nFROM patient_info\nORDER BY patient_id;"

display_query(query, title="NULL / BLANK NORMALIZATION PATTERNS · Step 2")


# Practice

1. Write a rule requiring explicit consent for an entire
   age/diagnosis condition. Use parentheses.
2. Build preferred contact: email -> mobile -> home -> fallback.
3. Classify bills into Unknown, No Charge, High-Value,
   Standard, and Low-Value.
4. Include statuses that are not definitely Paid using a
   NULL-safe predicate.
5. Classify completed admissions by length of stay.
CHECKLIST
[ ] I use parentheses when AND and OR are mixed.
[ ] I distinguish NULL from blank text and FALSE.
[ ] I understand UNKNOWN in WHERE conditions.
[ ] I order CASE branches from specific/high-priority to broad.
[ ] I test boundary values and NULL cases.

Try each task before looking back at the earlier examples.


In [ ]:
# Write your practice SQL here
practice_query = """

"""

# When ready:
# display_query(practice_query, title="Practice")


# Module 5 summary

Before moving on, make sure you can explain the main ideas in your own words and can describe the **grain** or row meaning of the important query results.

A good final check is:

1. What question did the query answer?
2. What did one input row represent?
3. What does one output row represent?
4. How did SQL change the data shape or meaning?
5. How would you validate that the result is correct?

## Next
Continue to **Module 6**.
